# ColabのL4 GPUでOllamaを立てる

ローカルの `local_llm` RAGアプリから、このColabノートブック上のOllama（L4 GPU）にAPI経由で接続するためのセットアップ。

**手順**
1. ランタイム → ランタイムのタイプを変更 → GPU（L4）を選択してから、上から順にセルを実行する
2. ngrokの無料アカウントを作り、ダッシュボードでauthtokenを控えておく（未取得なら ngrok.com で登録できる）
3. 最後のセルで表示される `OLLAMA_HOST` と `OLLAMA_API_KEY` を、ローカル側の `.env`（`.env.example` をコピーして作る）に設定する
4. ローカルで `streamlit run rag_chat_app.py` を起動する

**注意**
- Colabのランタイムはアイドルや時間経過で切断される。常時稼働のAPIではなく、使う時だけ起動するものとして扱うこと。
- ここで発行されるngrokのURLは誰でも到達できるが、`X-API-Key` ヘッダーが一致しないリクエストはプロキシが401で拒否する。それでもURLとAPIキーは他人に共有しないこと。

In [ ]:
# 1. Ollama本体のインストール
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# 2. Ollamaサーバーをバックグラウンドで起動し、応答を待つ
import subprocess
import time

import requests

ollama_proc = subprocess.Popen(
    ["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

for _ in range(30):
    try:
        requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
        print("Ollama起動確認OK")
        break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError("Ollamaの起動に失敗しました")

In [ ]:
# 3. モデルを取得する（埋め込み用 + 生成用2種類。時間がかかる）
!ollama pull bge-m3
!ollama pull qwen2.5:7b-instruct
!ollama pull llama3.1:8b

In [ ]:
# 4. 認証つきリバースプロキシに必要なパッケージ
!pip install -q fastapi "uvicorn[standard]" httpx pyngrok

In [ ]:
# 5. プロキシ用のAPIキーを発行する。この値をローカルの .env の OLLAMA_API_KEY に設定する
import secrets

OLLAMA_PROXY_API_KEY = secrets.token_urlsafe(24)
print("ローカル .env に設定する OLLAMA_API_KEY:")
print(OLLAMA_PROXY_API_KEY)

In [ ]:
# 6. リバースプロキシ本体を書き出す。
# X-API-Keyヘッダーを検証したうえで、パスをそのまま127.0.0.1:11434のOllamaへ
# ストリーミング転送する（チャットのSSEストリームもそのまま通す）。
from pathlib import Path

proxy_code = f'''
import httpx
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse, StreamingResponse

API_KEY = {OLLAMA_PROXY_API_KEY!r}
OLLAMA_BASE = "http://127.0.0.1:11434"

app = FastAPI()
_client = httpx.AsyncClient(base_url=OLLAMA_BASE, timeout=None)

_HOP_BY_HOP = {{"host", "x-api-key", "authorization", "content-length", "connection", "transfer-encoding"}}


@app.api_route("/{{path:path}}", methods=["GET", "POST"])
async def proxy(path: str, request: Request):
    if request.headers.get("x-api-key") != API_KEY:
        return JSONResponse({{"error": "unauthorized"}}, status_code=401)

    body = await request.body()
    upstream_req = _client.build_request(
        request.method,
        f"/{{path}}",
        params=request.query_params,
        content=body,
        headers=[(k, v) for k, v in request.headers.items() if k.lower() not in _HOP_BY_HOP],
    )
    upstream = await _client.send(upstream_req, stream=True)

    async def body_iter():
        async for chunk in upstream.aiter_raw():
            yield chunk
        await upstream.aclose()

    return StreamingResponse(
        body_iter(),
        status_code=upstream.status_code,
        headers={{k: v for k, v in upstream.headers.items() if k.lower() not in _HOP_BY_HOP}},
    )
'''

Path("ollama_proxy.py").write_text(proxy_code, encoding="utf-8")
print("ollama_proxy.py を書き出しました")

In [ ]:
# 7. プロキシをバックグラウンドで起動し、応答を待つ
proxy_proc = subprocess.Popen(
    ["uvicorn", "ollama_proxy:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

for _ in range(30):
    try:
        r = requests.get(
            "http://127.0.0.1:8000/api/tags",
            headers={"X-API-Key": OLLAMA_PROXY_API_KEY},
            timeout=2,
        )
        if r.status_code == 200:
            print("プロキシ起動確認OK")
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError("プロキシの起動に失敗しました")

In [ ]:
# 8. ngrokでプロキシ（8000番）を外部公開する。
# authtokenはngrokダッシュボード（https://dashboard.ngrok.com/get-started/your-authtoken）から取得する
from getpass import getpass

from pyngrok import conf, ngrok

conf.get_default().auth_token = getpass("ngrok authtoken: ")
public_url = ngrok.connect(8000, "http")
print("ローカル .env に設定する OLLAMA_HOST:")
print(public_url)

## ローカル側の設定

`local_llm/.env.example` を `.env` にコピーし、上のセルで表示された値を設定する。

```
OLLAMA_HOST=<上で表示された public_url>
OLLAMA_API_KEY=<セル5で表示された OLLAMA_PROXY_API_KEY>
```

保存したら、いつも通りローカルで起動する。

```powershell
myvenv313\Scripts\python.exe -m streamlit run rag_chat_app.py
```

サイドバーのモデル選択で `qwen2.5:7b-instruct` または `llama3.1:8b` を選べば、生成はColabのL4 GPUで行われる。

In [ ]:
# 9. 使い終わったら実行して後片付けする（トンネル・プロキシ・Ollamaを止める）
ngrok.disconnect(public_url.public_url)
proxy_proc.terminate()
ollama_proc.terminate()
print("停止しました")